# SE-ResNeXt-50-32x4d KL Grading

This is a single-task adaptation inspired by the Tiulpin et al. training recipe. It predicts the current five-class Kellgren-Lawrence grade only. Future progression, clinical features, and model stacking are intentionally excluded.

## Configuration Summary

| Item | Configuration |
| --- | --- |
| Task | One CNN head: KL grade 0-4 |
| Backbone | ImageNet-pretrained SE-ResNeXt-50-32x4d |
| Head | Global average pooling -> Dropout(0.50) -> Linear(2048 -> 5) |
| Input | Resize to 310, random crop to 300 during training; center-safe 300 crop for validation |
| Preprocessing | Percentile clipping 5th-99th, global [0,1] normalization, grayscale repeated to 3 channels |
| Augmentation | Gaussian noise, gamma correction, rotation +/-5 degrees, random 310 -> 300 crop |
| Loss | Five-class cross-entropy |
| Optimizer | Adam, learning rate 1e-3, weight decay 1e-4 |
| Training stages | Backbone frozen for 2 epochs, then all layers trainable for 20 epochs |
| Schedule | Learning rate reduced 10x at epoch 15 |
| Validation | Five-fold stratified subject-grouped cross-validation |
| Explainability | Post-hoc Grad-CAM from the final convolutional block |

The original paper used longitudinal OAI/MOST radiographs and progression targets. This notebook keeps only the paper-inspired image recipe and uses the local five-grade KL dataset.

Run every cell in a fresh Colab GPU runtime. The final sections display cross-validation metrics, a classification report, training curves, a confusion matrix, and Grad-CAM examples.


In [1]:
!pip -q install timm scikit-learn opencv-python-headless


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import copy
import gc
import json
import random
import re
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import accuracy_score, average_precision_score, classification_report, cohen_kappa_score, confusion_matrix, f1_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
DATA_ROOT = Path('/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224')
RUN_DIR = Path('/content/drive/MyDrive/Models/paper_se_resnext50_kl') / pd.Timestamp.utcnow().strftime('%Y-%m-%d_%H-%M-%S_UTC')
SOURCE_SIZE = 310
INPUT_SIZE = 300
BATCH_SIZE = 16
FROZEN_EPOCHS = 2
FULL_EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.50
NUM_WORKERS = 2

if not DATA_ROOT.exists():
    raise FileNotFoundError(f'KL dataset not found: {DATA_ROOT}')
RUN_DIR.mkdir(parents=True, exist_ok=False)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def subject_id_from_filename(path):
    # KneeXrayData names usually end in L or R. Remove only that side marker so
    # both knees from one patient stay in the same cross-validation fold.
    return re.sub(r'[LR]$', '', path.stem, flags=re.IGNORECASE)


records = []
for path in sorted(DATA_ROOT.glob('*/*/*.png')):
    try: grade = int(path.parent.name)
    except ValueError: continue
    if grade not in range(5): continue
    records.append({'image_path': str(path), 'subject_id': subject_id_from_filename(path), 'kl_grade': grade})
frame = pd.DataFrame(records)
if frame.empty: raise RuntimeError(f'No KL PNG files found under {DATA_ROOT}')
print('Rows:', len(frame), 'patient groups:', frame.subject_id.nunique(), 'device:', DEVICE)
print('Run directory:', RUN_DIR)


class KLDataset(torch.utils.data.Dataset):
    def __init__(self, rows, train):
        self.rows = rows.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        image = cv2.imread(str(row.image_path), cv2.IMREAD_GRAYSCALE)
        if image is None: raise RuntimeError(f'Cannot decode {row.image_path}')
        image = cv2.resize(image, (SOURCE_SIZE, SOURCE_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32)
        low, high = np.percentile(image, [5, 99])
        image = np.clip((image - low) / (high - low + 1e-8), 0, 1)
        if self.train:
            angle = random.uniform(-5, 5)
            matrix = cv2.getRotationMatrix2D((SOURCE_SIZE / 2, SOURCE_SIZE / 2), angle, 1.0)
            image = cv2.warpAffine(image, matrix, (SOURCE_SIZE, SOURCE_SIZE), borderMode=cv2.BORDER_REFLECT_101)
            image = np.power(np.clip(image, 0, 1), random.uniform(0.8, 1.2))
            image = np.clip(image + np.random.normal(0, 0.02, image.shape), 0, 1)
            left = random.randint(0, SOURCE_SIZE - INPUT_SIZE); top = random.randint(0, SOURCE_SIZE - INPUT_SIZE)
            image = image[top:top + INPUT_SIZE, left:left + INPUT_SIZE]
        else:
            image = image[5:305, 5:305]
        return torch.from_numpy(np.repeat(image[None], 3, axis=0).copy()).float(), int(row.kl_grade)


class SEResNeXtKLCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('seresnext50_32x4d', pretrained=True, num_classes=0, global_pool='avg')
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(self.backbone.num_features, 5))

    @property
    def gradcam_target_layer(self):
        return self.backbone.layer4

    def forward(self, images):
        return self.classifier(self.backbone(images))


def evaluate(model, loader):
    model.eval(); labels_all = []; probabilities_all = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images)
            labels_all.extend(labels.numpy())
            probabilities_all.extend(F.softmax(logits, dim=1).cpu().numpy())
    labels_all = np.asarray(labels_all); probabilities_all = np.asarray(probabilities_all); predictions_all = probabilities_all.argmax(1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels_all, predictions_all, average='macro', zero_division=0)
    try:
        macro_ap = average_precision_score(np.eye(5)[labels_all], probabilities_all, average='macro')
        auc_ovr = roc_auc_score(labels_all, probabilities_all, multi_class='ovr')
    except ValueError:
        macro_ap, auc_ovr = float('nan'), float('nan')
    return {
        'accuracy': float(accuracy_score(labels_all, predictions_all)),
        'qwk': float(cohen_kappa_score(labels_all, predictions_all, weights='quadratic')),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
        'macro_ap': float(macro_ap),
        'auc_ovr': float(auc_ovr),
        'labels': labels_all,
        'probabilities': probabilities_all,
        'predictions': predictions_all,
    }


def train_fold(train_rows, val_rows, fold):
    model = SEResNeXtKLCNN().to(DEVICE)
    for parameter in model.backbone.parameters(): parameter.requires_grad = False
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    train_loader = torch.utils.data.DataLoader(KLDataset(train_rows, True), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
    val_loader = torch.utils.data.DataLoader(KLDataset(val_rows, False), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
    history = []; best_state = None; best_score = -float('inf')
    use_amp = DEVICE.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    for epoch in range(1, FROZEN_EPOCHS + FULL_EPOCHS + 1):
        if epoch == FROZEN_EPOCHS + 1:
            for parameter in model.backbone.parameters(): parameter.requires_grad = True
        if epoch == 15:
            for group in optimizer.param_groups: group['lr'] = LEARNING_RATE * 0.1
        model.train(); total_loss = 0.0; sample_count = 0
        for images, labels in train_loader:
            # Explicitly move both tensors before computing cross-entropy.
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            total_loss += loss.item() * labels.size(0); sample_count += labels.size(0)
        metrics = evaluate(model, val_loader)
        row = {'fold': fold, 'epoch': epoch, 'train_loss': total_loss / sample_count, **{key: metrics[key] for key in ('accuracy', 'qwk', 'macro_f1', 'macro_ap')}}
        history.append(row); print(row)
        selection = metrics['qwk'] + metrics['macro_f1']
        if selection > best_score:
            best_score = selection; best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    selected = evaluate(model, val_loader)
    checkpoint_path = RUN_DIR / f'fold_{fold}_best.pth'
    torch.save({'model_state_dict': best_state, 'paper_inspired': 'tiulpin_2019', 'task': 'kl_grade_0_to_4', 'fold': fold, 'config': {'batch_size': BATCH_SIZE, 'frozen_epochs': FROZEN_EPOCHS, 'full_epochs': FULL_EPOCHS, 'learning_rate': LEARNING_RATE, 'lr_drop_epoch': 15, 'weight_decay': WEIGHT_DECAY, 'dropout': DROPOUT}}, checkpoint_path)
    del model, optimizer, train_loader, val_loader, best_state
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return history, selected, checkpoint_path


splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
all_history = []; oof_rows = []; fold_summaries = []; fold_validation_rows = {}
for fold, (train_idx, val_idx) in enumerate(splitter.split(frame, frame.kl_grade, groups=frame.subject_id), 1):
    train_rows, val_rows = frame.iloc[train_idx], frame.iloc[val_idx]
    fold_history, selected, checkpoint_path = train_fold(train_rows, val_rows, fold)
    all_history.extend(fold_history); fold_validation_rows[fold] = val_rows.reset_index(drop=True)
    fold_summaries.append({'fold': fold, 'checkpoint': str(checkpoint_path), **{key: selected[key] for key in ('accuracy', 'qwk', 'macro_precision', 'macro_recall', 'macro_f1', 'macro_ap', 'auc_ovr')}})
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    for row, label, prediction, probability in zip(val_rows.itertuples(), selected['labels'], selected['predictions'], selected['probabilities']):
        oof_rows.append({'fold': fold, 'image_path': row.image_path, 'true_grade': int(label), 'predicted_grade': int(prediction), **{f'probability_{grade}': float(probability[grade]) for grade in range(5)}})

history_df = pd.DataFrame(all_history); oof_df = pd.DataFrame(oof_rows); fold_summary_df = pd.DataFrame(fold_summaries)
history_df.to_csv(RUN_DIR / 'training_history.csv', index=False)
oof_df.to_csv(RUN_DIR / 'oof_predictions.csv', index=False)
fold_summary_df.to_csv(RUN_DIR / 'fold_metrics.csv', index=False)
with open(RUN_DIR / 'run_config.json', 'w') as handle:
    json.dump({'paper_inspired': 'tiulpin_2019', 'model': 'SE-ResNeXt-50-32x4d', 'task': 'kl_grade_0_to_4', 'input_size': INPUT_SIZE, 'loss': 'cross_entropy', 'optimizer': 'Adam', 'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY, 'frozen_epochs': FROZEN_EPOCHS, 'full_epochs': FULL_EPOCHS, 'lr_drop_epoch': 15}, handle, indent=2)
print('Five-fold single-head KL training complete:', RUN_DIR)


Mounted at /content/drive
Rows: 9786 patient groups: 5656 device: cuda
Run directory: /content/drive/MyDrive/Models/paper_se_resnext50_kl/2026-08-09_06-51-17_UTC


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  111MB            

model.safetensors: downloading bytes:           |  0.00B            

## Dataset and target

This notebook uses the local `kneeKL224` folder. Labels come from the grade directory (`0` through `4`). It does not require progression labels, clinical features, OAI/MOST follow-up data, or a second prediction head. Subject grouping uses the leading filename token as a grouping key; confirm that convention matches your files before interpreting the folds as patient-wise.

The paper's original ROI alignment and standardized 310x310 image acquisition are approximated here from the available cropped KL images by percentile normalization and resize. The training objective remains current KL grading only.



## Completion

The preceding code trains five subject-grouped folds and writes one selected checkpoint per fold under `RUN_DIR`. The following cells display out-of-fold metrics, plots, and Grad-CAM examples directly in this notebook.


## Cross-Validation Evaluation Dashboard

Out-of-fold predictions combine only the validation prediction from each fold. This gives each image one prediction from a model that did not train on that image.


In [ ]:
oof_labels = oof_df['true_grade'].to_numpy()
oof_predictions = oof_df['predicted_grade'].to_numpy()
oof_probabilities = oof_df[[f'probability_{grade}' for grade in range(5)]].to_numpy()
precision, recall, f1, _ = precision_recall_fscore_support(oof_labels, oof_predictions, average='macro', zero_division=0)
try:
    oof_ap = average_precision_score(np.eye(5)[oof_labels], oof_probabilities, average='macro')
    oof_auc = roc_auc_score(oof_labels, oof_probabilities, multi_class='ovr')
except ValueError:
    oof_ap, oof_auc = float('nan'), float('nan')
oof_metrics = {
    'Accuracy': accuracy_score(oof_labels, oof_predictions),
    'QWK': cohen_kappa_score(oof_labels, oof_predictions, weights='quadratic'),
    'Macro precision': precision,
    'Macro recall': recall,
    'Macro F1': f1,
    'Macro AP': oof_ap,
    'Macro ROC AUC (OvR)': oof_auc,
}
display(pd.DataFrame.from_dict(oof_metrics, orient='index', columns=['Out-of-fold result']).style.format('{:.4f}'))
print('Classification report')
display(pd.DataFrame(classification_report(oof_labels, oof_predictions, labels=range(5), output_dict=True, zero_division=0)).T.style.format('{:.4f}'))
display(fold_summary_df.drop(columns='checkpoint').style.format('{:.4f}'))

matrix = confusion_matrix(oof_labels, oof_predictions, labels=range(5))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for fold, fold_history in history_df.groupby('fold'):
    axes[0].plot(fold_history['epoch'], fold_history['train_loss'], alpha=0.65, label=f'Fold {fold}')
axes[0].set(title='Training loss by fold', xlabel='Epoch', ylabel='Cross-entropy loss')
axes[0].legend(); axes[0].grid(alpha=0.25)
image = axes[1].imshow(matrix, cmap='Blues')
axes[1].set(title='Out-of-fold confusion matrix', xlabel='Predicted KL grade', ylabel='True KL grade', xticks=range(5), yticks=range(5))
for row in range(5):
    for column in range(5): axes[1].text(column, row, matrix[row, column], ha='center', va='center')
fig.colorbar(image, ax=axes[1], fraction=0.046)
plt.tight_layout(); plt.savefig(RUN_DIR / 'evaluation_dashboard.png', dpi=180, bbox_inches='tight'); plt.show()
pd.DataFrame(matrix, index=range(5), columns=range(5)).to_csv(RUN_DIR / 'oof_confusion_matrix.csv')
with open(RUN_DIR / 'oof_metrics.json', 'w') as handle: json.dump({key: float(value) for key, value in oof_metrics.items()}, handle, indent=2)


## Grad-CAM Examples

For a readable notebook gallery, Grad-CAM is generated from the selected fold-1 model on five fold-1 validation images, one from each true KL grade. The map always explains the predicted class.


In [ ]:
def se_resnext_gradcam(model, input_tensor, class_index):
    captured = {}

    def capture(_module, _inputs, output):
        captured['activation'] = output

    handle = model.gradcam_target_layer.register_forward_hook(capture)
    try:
        model.eval(); model.zero_grad(set_to_none=True)
        logits = model(input_tensor)
        activation = captured['activation']
        gradients = torch.autograd.grad(logits[0, class_index], activation)[0]
        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * activation).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(INPUT_SIZE, INPUT_SIZE), mode='bilinear', align_corners=False)[0, 0]
    finally:
        handle.remove()
    cam = cam.detach().cpu().numpy()
    return cam / cam.max() if cam.max() > 1e-8 else np.zeros_like(cam)


def display_image_for_gradcam(image_path):
    # Match the validation image geometry, but display the original intensity rather
    # than the normalized tensor used by the model.
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None: raise RuntimeError(f'Cannot decode {image_path}')
    image = cv2.resize(image, (SOURCE_SIZE, SOURCE_SIZE), interpolation=cv2.INTER_AREA)
    return image[5:305, 5:305]


def show_gradcam(axis, display_image, cam, title):
    # Transparent low values reproduce the readable notebook style used elsewhere.
    visible_cam = np.ma.masked_where(cam <= 0.05, cam)
    axis.imshow(display_image, cmap='gray', vmin=0, vmax=255)
    axis.imshow(visible_cam, cmap='jet', alpha=0.42, vmin=0, vmax=1)
    axis.set_title(title); axis.axis('off')


cam_model = SEResNeXtKLCNN().to(DEVICE)
cam_model.load_state_dict(torch.load(RUN_DIR / 'fold_1_best.pth', map_location=DEVICE, weights_only=False)['model_state_dict'])
cam_rows = fold_validation_rows[1].groupby('kl_grade', group_keys=False).head(1).sort_values('kl_grade')
cam_dataset = KLDataset(cam_rows, train=False)
fig, axes = plt.subplots(len(cam_rows), 2, figsize=(9, 4 * len(cam_rows)))
for row_index, row in enumerate(cam_rows.itertuples()):
    tensor, true_grade = cam_dataset[row_index]
    tensor = tensor[None].to(DEVICE)
    with torch.no_grad():
        logits = cam_model(tensor); predicted_grade = int(logits.argmax(1).item()); confidence = float(F.softmax(logits, 1)[0, predicted_grade].item())
    cam = se_resnext_gradcam(cam_model, tensor, predicted_grade)
    display_image = display_image_for_gradcam(row.image_path)
    axes[row_index, 0].imshow(display_image, cmap='gray'); axes[row_index, 0].set_title(f'True KL {true_grade}'); axes[row_index, 0].axis('off')
    show_gradcam(axes[row_index, 1], display_image, cam, f'Grad-CAM: predicted KL {predicted_grade} ({confidence:.1%})')
plt.tight_layout(); plt.savefig(RUN_DIR / 'gradcam_by_true_grade.png', dpi=180, bbox_inches='tight'); plt.show()
